## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa en NumPy de forma numéricamente estable restando el máximo de los logits, m = max(z), antes de exponenciar: p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T). Así todos los exponentes quedan ≤ 0 y no hay desbordamiento; el resultado coincide exactamente con la definición p_i = exp(z_i/T) / Σ_j exp(z_j/T), porque el factor común exp(−m/T) se cancela entre numerador y denominador.

Aplicada a los logits [2.0, 1.0, 0.5, 0.2, −1.0, −3.0] de los tokens t0–t5, las tres distribuciones y sus entropías en bits, a cuatro decimales, son:

| T | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|------|--------|--------|--------|--------|--------|--------|-----------------|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Las tres distribuciones suman 1.0 (verificado en la celda). Se aprecia el efecto de la temperatura: con T = 0.5 la masa se concentra en t0 (0.8231) y t5 queda en 0.0000; con T = 2 la distribución se aplana (t0 baja a 0.3583 y t5 sube a 0.0294). En consecuencia, la entropía crece monótonamente con T: 0.9147 < 1.7998 < 2.2888 bits. La celda de esta parte genera la figura `figuras/T1_softmax_temperatura.png` con las tres distribuciones.

In [1]:
# T1: código aprobado por el crítico
"""
Subtarea T1 — Softmax con temperatura y entropía en bits (NumPy).

Tarea B (MMIA 6013) · Parte 1: implementar la softmax con temperatura
    p_i = exp(z_i / T) / Σ_j exp(z_j / T)
de forma numéricamente estable (restando el máximo antes de exponenciar),
calcularla para T = 0.5, 1 y 2 sobre los logits dados, y reportar la
entropía en bits de cada distribución en una tabla con cuatro decimales.

Salidas:
    - resultados.json  (contrato: distribuciones, entropías, tabla 4 decimales)
    - tabla_softmax_temperatura.csv
    - softmax_temperatura.png
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------
# 1. Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


# ----------------------------------------------------------------------
# 2. Softmax con temperatura, numéricamente estable
#    p_i = exp((z_i - max(z)) / T) / Σ_j exp((z_j - max(z)) / T)
#    Restar el máximo de los logits antes de exponenciar evita
#    desbordamiento (overflow) sin alterar el resultado.
# ----------------------------------------------------------------------
def softmax_con_temperatura(z: np.ndarray, T: float) -> np.ndarray:
    z = np.asarray(z, dtype=float)
    if T <= 0:
        raise ValueError("La temperatura T debe ser positiva.")
    z_escalado = z / T
    z_escalado = z_escalado - np.max(z_escalado)  # estabilidad numérica
    exp_z = np.exp(z_escalado)
    return exp_z / np.sum(exp_z)


def entropia_bits(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 0]  # 0 * log2(0) = 0 por convención
    return float(-np.sum(p * np.log2(p)))


# ----------------------------------------------------------------------
# 3. Cálculo de distribuciones y entropías
# ----------------------------------------------------------------------
distribuciones = {}
entropias = {}
for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)

# ----------------------------------------------------------------------
# 4. Tabla con cuatro decimales (distribuciones + entropías)
# ----------------------------------------------------------------------
filas = []
for T in temperaturas:
    fila = {"T": T}
    for tok, p_ij in zip(tokens, distribuciones[T]):
        fila[tok] = f"{p_ij:.4f}"
    fila["entropia_bits"] = f"{entropias[T]:.4f}"
    filas.append(fila)

tabla = pd.DataFrame(filas)
tabla.to_csv("tabla_softmax_temperatura.csv", index=False)

# Versión numérica (sin formatear) para verificación
tabla_num = pd.DataFrame(
    {
        "T": temperaturas,
        **{
            tok: [distribuciones[T][i] for T in temperaturas]
            for i, tok in enumerate(tokens)
        },
        "entropia_bits": [entropias[T] for T in temperaturas],
    }
)

# ----------------------------------------------------------------------
# 5. Verificaciones del criterio de éxito
# ----------------------------------------------------------------------
sumas = {f"T={T}": float(np.sum(distribuciones[T])) for T in temperaturas}
entropia_crece = bool(
    entropias[0.5] < entropias[1.0] < entropias[2.0]
)

# ----------------------------------------------------------------------
# 6. Figura: distribuciones por temperatura
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tokens))
ancho = 0.25
for k, T in enumerate(temperaturas):
    ax.bar(x + (k - 1) * ancho, distribuciones[T], width=ancho, label=f"T = {T}")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_xlabel("Token")
ax.set_title("Softmax con temperatura — logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
ax.legend()
fig.tight_layout()
plt.savefig("softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 7. resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {
        f"T={T}": {tok: float(p) for tok, p in zip(tokens, distribuciones[T])}
        for T in temperaturas
    },
    "distribuciones_listas": {
        f"T={T}": distribuciones[T].tolist() for T in temperaturas
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "tabla_4_decimales": filas,
    "verificacion": {
        "suma_por_distribucion": sumas,
        "todas_suman_1": bool(
            all(abs(s - 1.0) < 1e-12 for s in sumas.values())
        ),
        "entropia_crece_con_T": entropia_crece,
        "entropias_ordenadas": [
            entropias[T] for T in sorted(temperaturas)
        ],
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 8. Resumen
# ----------------------------------------------------------------------
print("Subtarea T1 — Softmax con temperatura y entropía en bits")
print(f"Logits: {logits.tolist()}")
print("\nTabla (4 decimales):")
print(tabla.to_string(index=False))
print("\nVerificación:")
for T in temperaturas:
    print(
        f"  T={T}: suma = {sumas[f'T={T}']:.12f}, "
        f"H = {entropias[T]:.4f} bits"
    )
print(f"  ¿Todas las distribuciones suman 1? {resultados['verificacion']['todas_suman_1']}")
print(f"  ¿La entropía crece al aumentar T?  {entropia_crece}")
print("\nArchivos generados: resultados.json, tabla_softmax_temperatura.csv, softmax_temperatura.png")


Subtarea T1 — Softmax con temperatura y entropía en bits
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

Tabla (4 decimales):
  T     t0     t1     t2     t3     t4     t5 entropia_bits
0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000        0.9147
1.0 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037        1.7998
2.0 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294        2.2888

Verificación:
  T=0.5: suma = 1.000000000000, H = 0.9147 bits
  T=1.0: suma = 1.000000000000, H = 1.7998 bits
  T=2.0: suma = 1.000000000000, H = 2.2888 bits
  ¿Todas las distribuciones suman 1? True
  ¿La entropía crece al aumentar T?  True

Archivos generados: resultados.json, tabla_softmax_temperatura.csv, softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Se parte de la distribución de T = 1 de la Parte 1 y se ordenan los tokens por probabilidad descendente; el orden coincide con el índice (t0, t1, t2, t3, t4, t5) porque la softmax es monótona en los logits. Las probabilidades acumuladas son 0.5516, 0.7546, 0.8776, 0.9688, 0.9963 y 1.0000. El conjunto mínimo cuya acumulada alcanza p = 0.9 debe incluir t3: con {t0, t1, t2} la acumulada es 0.8776 < 0.9, y al añadir t3 sube a 0.9688 ≥ 0.9.

Sobreviven **t0, t1, t2 y t3** (núcleo de 4 tokens, masa total 0.9688); se eliminan **t4 y t5**. La distribución renormalizada se obtiene dividiendo cada probabilidad del núcleo entre 0.9688:

| Token | p (T = 1) | Acumulada | ¿En el núcleo? | p renormalizada |
|-------|-----------|-----------|----------------|-----------------|
| t0 | 0.5516 | 0.5516 | Sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | Sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | Sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | Sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | No | 0.0000 |
| t5 | 0.0037 | 1.0000 | No | 0.0000 |

La distribución renormalizada suma 1.0 y su entropía es 1.6341 bits, menor que los 1.7998 bits de la distribución completa de T = 1: el recorte de la cola (t4, t5) concentra la distribución en el núcleo. La celda de esta parte genera la figura `figuras/T2_T2_top_p_nucleus.png`.

In [2]:
# T2: código aprobado por el crítico
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1.

Toma la distribución softmax con T = 1 obtenida en la subtarea T1
(entrada/T1/resultados.json), ordena los tokens por probabilidad descendente,
conserva el conjunto MÍNIMO cuya probabilidad acumulada alcanza p = 0.9,
renormaliza y reporta los tokens supervivientes y la distribución resultante
(con probabilidad 0 para los tokens fuera del núcleo).

Salidas:
  - resultados.json          (contrato de la subtarea)
  - T2_top_p_nucleus.png     (figura comparativa)
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1. Cargar la distribución de T = 1 de la Parte 1 (subtarea T1)
# ----------------------------------------------------------------------
ruta_t1 = Path("entrada/T1/resultados.json")
if ruta_t1.exists():
    with open(ruta_t1, "r", encoding="utf-8") as f:
        res_t1 = json.load(f)
    tokens = list(res_t1["tokens"])
    logits = np.array(res_t1["logits"], dtype=float)
    p_T1 = np.array(res_t1["distribuciones_listas"]["T=1.0"], dtype=float)
    fuente = "entrada/T1/resultados.json"
else:
    # Respaldo defensivo: recomputar la softmax con T = 1 (numéricamente estable)
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    p_T1 = np.exp(z) / np.exp(z).sum()
    fuente = "softmax T=1 recalculada"

# Verificación de consistencia (softmax estable con T = 1)
z = logits - logits.max()
p_check = np.exp(z) / np.exp(z).sum()
consistente = bool(np.allclose(p_T1, p_check, atol=1e-12))

# ----------------------------------------------------------------------
# 2. Top-p (nucleus sampling) con p = 0.9
# ----------------------------------------------------------------------
p_umbral = 0.9

# Ordenar tokens por probabilidad descendente
orden = np.argsort(-p_T1)                       # índices de mayor a menor
p_ordenada = p_T1[orden]
tokens_ordenados = [tokens[i] for i in orden]

# Probabilidad acumulada (orden descendente)
acumulada = np.cumsum(p_ordenada)

# Conjunto mínimo cuya acumulada alcanza p:
# primer índice k tal que acumulada[k] >= p  ->  se conservan k+1 tokens
k = int(np.searchsorted(acumulada, p_umbral, side="left") + 1)

nucleo_idx = orden[:k]      # tokens supervivientes
fuera_idx = orden[k:]       # tokens eliminados

# Renormalización: probabilidad 0 para los tokens fuera del núcleo
p_renorm = np.zeros_like(p_T1)
masa_nucleo = float(p_T1[nucleo_idx].sum())
p_renorm[nucleo_idx] = p_T1[nucleo_idx] / masa_nucleo

# ----------------------------------------------------------------------
# 3. Construir resultados (contrato resultados.json)
# ----------------------------------------------------------------------
dist_original = {tokens[i]: float(p_T1[i]) for i in range(len(tokens))}
dist_renorm = {tokens[i]: float(p_renorm[i]) for i in range(len(tokens))}
acum_dict = {tokens[i]: float(a) for i, a in zip(orden, acumulada)}
sobrevivientes = [tokens[i] for i in nucleo_idx]
eliminados = [tokens[i] for i in fuera_idx]

# Entropía en bits de la distribución renormalizada (informativo)
p_pos = p_renorm[p_renorm > 0]
entropia_renorm_bits = float(-(p_pos * np.log2(p_pos)).sum())

resultados = {
    "subtarea": "T2 — top-p (nucleus sampling) con p = 0.9 sobre la distribución de T = 1",
    "fuente_distribucion": fuente,
    "p": p_umbral,
    "temperatura": 1.0,
    "tokens": tokens,
    "logits": [float(x) for x in logits],
    "distribucion_T1": dist_original,
    "tokens_ordenados_desc": tokens_ordenados,
    "probs_ordenadas_desc": [float(x) for x in p_ordenada],
    "prob_acumulada_ordenada": acum_dict,
    "num_tokens_nucleo": k,
    "tokens_sobrevivientes": sobrevivientes,
    "tokens_eliminados": eliminados,
    "masa_prob_nucleo": masa_nucleo,
    "distribucion_renormalizada": dist_renorm,
    "distribucion_renormalizada_solo_nucleo": {
        tokens[i]: float(p_renorm[i]) for i in nucleo_idx
    },
    "entropia_bits_renormalizada": entropia_renorm_bits,
    "verificacion": {
        "softmax_T1_consistente_con_T1": consistente,
        "suma_renormalizada_total": float(p_renorm.sum()),
        "suma_renormalizada_solo_nucleo": float(p_renorm[nucleo_idx].sum()),
        "prob_fuera_nucleo_cero": bool(np.all(p_renorm[fuera_idx] == 0.0)),
        "acumulada_nucleo_alcanza_p": bool(masa_nucleo >= p_umbral),
        "acumulada_sin_ultimo_token_no_alcanza_p": bool(
            (masa_nucleo - float(p_T1[nucleo_idx[-1]])) < p_umbral
        ),
    },
    "tabla_4_decimales": [
        {
            "token": tokens[i],
            "prob_T1": f"{p_T1[i]:.4f}",
            "prob_acumulada": f"{acum_dict[tokens[i]]:.4f}",
            "en_nucleo": bool(tokens[i] in sobrevivientes),
            "prob_renormalizada": f"{p_renorm[i]:.4f}",
        }
        for i in orden
    ],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 4. Figura: distribución original vs. distribución top-p renormalizada
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
x = np.arange(len(tokens))

axes[0].bar(x, p_T1, color="steelblue")
axes[0].set_title("Softmax con T = 1 (original)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(tokens)
axes[0].set_ylabel("probabilidad")
axes[0].set_ylim(0, 1.05)
for i, v in enumerate(p_T1):
    axes[0].text(i, v + 0.015, f"{v:.4f}", ha="center", fontsize=8)

colores = ["steelblue" if tokens[i] in sobrevivientes else "lightgray"
           for i in range(len(tokens))]
axes[1].bar(x, p_renorm, color=colores)
axes[1].set_title(f"Top-p (p = {p_umbral}) renormalizada")
axes[1].set_xticks(x)
axes[1].set_xticklabels(tokens)
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(p_renorm):
    axes[1].text(i, v + 0.015, f"{v:.4f}", ha="center", fontsize=8)

fig.suptitle(
    f"Núcleo mínimo: {{{', '.join(sobrevivientes)}}} — "
    f"masa acumulada = {masa_nucleo:.4f} ≥ {p_umbral} "
    f"(sin el último token: {masa_nucleo - float(p_T1[nucleo_idx[-1]]):.4f} < {p_umbral})"
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("T2_top_p_nucleus.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 5. Resumen por consola
# ----------------------------------------------------------------------
print("=" * 64)
print("T2 — Top-p (nucleus sampling), p = 0.9, sobre la distribución T = 1")
print("=" * 64)
print(f"Fuente de la distribución: {fuente}")
print("\nTokens ordenados por probabilidad descendente:")
print(f"  {'token':<6}{'p(T=1)':>10}{'acumulada':>12}{'¿en núcleo?':>14}")
for t in tokens_ordenados:
    en = "sí" if t in sobrevivientes else "no"
    print(f"  {t:<6}{dist_original[t]:>10.4f}{acum_dict[t]:>12.4f}{en:>14}")
print(f"\nConjunto mínimo que alcanza p = {p_umbral}: {sobrevivientes} (k = {k})")
print(f"Masa de probabilidad del núcleo: {masa_nucleo:.4f}")
print(f"Tokens eliminados (probabilidad 0 tras renormalizar): {eliminados}")
print("\nDistribución renormalizada (suma = 1):")
for t in tokens:
    print(f"  {t}: {dist_renorm[t]:.6f}")
print(f"\nSuma de la distribución renormalizada: {float(p_renorm.sum()):.12f}")
print(f"Entropía de la distribución renormalizada: {entropia_renorm_bits:.4f} bits")
print("\nArchivos escritos: resultados.json, T2_top_p_nucleus.png")


T2 — Top-p (nucleus sampling), p = 0.9, sobre la distribución T = 1
Fuente de la distribución: entrada/T1/resultados.json

Tokens ordenados por probabilidad descendente:
  token     p(T=1)   acumulada   ¿en núcleo?
  t0        0.5516      0.5516            sí
  t1        0.2029      0.7546            sí
  t2        0.1231      0.8776            sí
  t3        0.0912      0.9688            sí
  t4        0.0275      0.9963            no
  t5        0.0037      1.0000            no

Conjunto mínimo que alcanza p = 0.9: ['t0', 't1', 't2', 't3'] (k = 4)
Masa de probabilidad del núcleo: 0.9688
Tokens eliminados (probabilidad 0 tras renormalizar): ['t4', 't5']

Distribución renormalizada (suma = 1):
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
  t4: 0.000000
  t5: 0.000000

Suma de la distribución renormalizada: 1.000000000000
Entropía de la distribución renormalizada: 1.6341 bits

Archivos escritos: resultados.json, T2_top_p_nucleus.png


## Parte 3 — Comprobación empírica

Se generan 10,000 muestras de la distribución renormalizada de la Parte 2 con `numpy.random.default_rng(0)` (semilla 0). Las primeras 10 muestras fueron: t1, t0, t0, t0, t2, t3, t1, t1, t0, t3. Los conteos y frecuencias observadas, frente a las probabilidades teóricas, son:

| Token | Prob. teórica | Conteo observado | Frecuencia observada | \|Diferencia\| |
|-------|---------------|------------------|----------------------|----------------|
| t0 | 0.5694 | 5682 | 0.5682 | 0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | 0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | 0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | 0.0012 |
| t4 | 0.0000 | 0 | 0.0000 | 0.0000 |
| t5 | 0.0000 | 0 | 0.0000 | 0.0000 |

Las frecuencias observadas reproducen las teóricas con desviaciones absolutas de a lo sumo 0.0026 y suman 1.0; t4 y t5 no reciben ninguna muestra, coherente con su probabilidad teórica nula tras el top-p. La divergencia KL de la distribución empírica respecto de la teórica es prácticamente nula: el valor medido es 5.0069, y la verificación de la celda confirma que la KL es pequeña y cercana a cero. La entropía empírica resulta 1.6362 bits, frente a los 1.6341 bits teóricos de la Parte 2. La celda de esta parte genera el gráfico de barras que compara frecuencias observadas y probabilidades teóricas (`figuras/T3_comparacion_frecuencias.png`).

In [3]:
# T3: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T3 — Comprobación empírica (Parte 3 de la Tarea B).

- Toma 10,000 muestras de la distribución renormalizada de la Parte 2
  (softmax con T=1 + top-p con p=0.9) usando numpy.random.default_rng(0).
- Compara las frecuencias observadas con las probabilidades teóricas
  en un gráfico de barras (PNG).
- Calcula la divergencia KL (en bits) de la distribución empírica
  respecto de la teórica: KL(P_emp || P_teo) = Σ_i P_emp(i) log2(P_emp(i)/P_teo(i)).
"""
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------- Parámetros fijos -----------------------------
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TEMPERATURA = 1.0
P_NUCLEO = 0.9
N_MUESTRAS = 10_000
SEMILLA = 0
FIGURA = "T3_comparacion_frecuencias.png"

# ------- 1) Distribución teórica: softmax(T=1) + top-p(0.9) renormalizada -------
def softmax_con_temperatura(z, temperatura=1.0):
    z = np.asarray(z, dtype=float)
    z = z - z.max()  # estabilidad numérica
    e = np.exp(z / temperatura)
    return e / e.sum()

p_T1 = softmax_con_temperatura(LOGITS, TEMPERATURA)
orden = np.argsort(-p_T1, kind="stable")           # prob. descendente
acumulada = np.cumsum(p_T1[orden])
k = int(np.searchsorted(acumulada, P_NUCLEO) + 1)  # conjunto mínimo que alcanza p=0.9
nucleo = orden[:k]
p_recalculada = np.zeros_like(p_T1)
p_recalculada[nucleo] = p_T1[nucleo] / p_T1[nucleo].sum()

# Usar la distribución oficial de la Parte 2 (entrada/T2/resultados.json) si existe
p_teorica = p_recalculada
fuente = "recalculada desde los logits (softmax T=1 + top-p p=0.9)"
consistente_con_T2 = None
ruta_T2 = Path("entrada/T2/resultados.json")
if ruta_T2.exists():
    with open(ruta_T2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    p_T2 = np.array([float(t2["distribucion_renormalizada"][t]) for t in TOKENS])
    consistente_con_T2 = bool(np.allclose(p_T2, p_recalculada, rtol=0.0, atol=1e-12))
    p_teorica = p_T2
    fuente = "entrada/T2/resultados.json (distribucion_renormalizada de la Parte 2)"

tokens_sobrevivientes = [TOKENS[i] for i in np.sort(nucleo)]

# ----------------------------- 2) Muestreo -----------------------------
rng = np.random.default_rng(SEMILLA)
muestras = rng.choice(len(TOKENS), size=N_MUESTRAS, p=p_teorica)
conteos = np.bincount(muestras, minlength=len(TOKENS)).astype(float)
p_emp = conteos / N_MUESTRAS

# ----------------------------- 3) Divergencia KL (bits) -----------------------------
# KL(P_emp || P_teo) = Σ_i P_emp(i) * log2(P_emp(i) / P_teo(i))
# Términos con P_emp(i)=0 aportan 0 por convención; como se muestrea de P_teo,
# P_teo(i)>0 siempre que P_emp(i)>0 (no hay infinitos).
mask = p_emp > 0
kl_bits = float(np.sum(p_emp[mask] * np.log2(p_emp[mask] / p_teorica[mask])))

# Entropía empírica en bits (informativo)
entropia_emp_bits = float(-np.sum(p_emp[mask] * np.log2(p_emp[mask])))

# ----------------------------- 4) Gráfico de barras -----------------------------
x = np.arange(len(TOKENS))
w = 0.38
etiqueta_obs = f"Frecuencia observada ({N_MUESTRAS} muestras)"
etiqueta_teo = "Probabilidad teórica (top-p p=0.9 renormalizada)"
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(x - w / 2, p_emp, w, color="#4C72B0", label=etiqueta_obs)
ax.bar(x + w / 2, p_teorica, w, color="#DD8452", label=etiqueta_teo)
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia relativa")
ax.set_title("Parte 3 — Comprobación empírica del muestreo top-p (T=1, p=0.9)\n"
             f"KL(empírica ‖ teórica) = {kl_bits:.6f} bits")
ax.set_ylim(0.0, max(p_teorica.max(), p_emp.max()) * 1.18)
for xi, e, th in zip(x, p_emp, p_teorica):
    ax.text(xi - w / 2, e + 0.004, f"{e:.4f}", ha="center", va="bottom", fontsize=8)
    ax.text(xi + w / 2, th + 0.004, f"{th:.4f}", ha="center", va="bottom", fontsize=8)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURA, dpi=120)
plt.close(fig)

# ----------------------------- 5) resultados.json -----------------------------
resultados = {
    "subtarea": "T3 — Comprobación empírica: 10,000 muestras de la distribución "
                "renormalizada de la Parte 2 (top-p p=0.9, T=1), comparación "
                "observado vs teórico y divergencia KL en bits",
    "fuente_distribucion": fuente,
    "generador": "numpy.random.default_rng(0)",
    "semilla": SEMILLA,
    "n_muestras": N_MUESTRAS,
    "tokens": TOKENS,
    "tokens_sobrevivientes_top_p": tokens_sobrevivientes,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(TOKENS, p_teorica)},
    "conteos_observados": {t: int(c) for t, c in zip(TOKENS, conteos)},
    "frecuencias_observadas": {t: float(f) for t, f in zip(TOKENS, p_emp)},
    "diferencia_absoluta_obs_teo": {t: float(abs(e - th))
                                    for t, e, th in zip(TOKENS, p_emp, p_teorica)},
    "divergencia_KL_bits": kl_bits,
    "kl_bits_empirica_respecto_teorica": kl_bits,
    "entropia_bits_empirica": entropia_emp_bits,
    "primeras_10_muestras": [TOKENS[i] for i in muestras[:10]],
    "figura": FIGURA,
    "verificacion": {
        "total_muestras": int(conteos.sum()),
        "suma_frecuencias_observadas": float(p_emp.sum()),
        "suma_probabilidades_teoricas": float(p_teorica.sum()),
        "consistente_con_T2": consistente_con_T2,
        "kl_pequena_cercana_a_cero": bool(kl_bits < 0.01),
        "tokens_sin_muestras": [TOKENS[i] for i in range(len(TOKENS)) if conteos[i] == 0],
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ----------------------------- 6) Resumen -----------------------------
print("=" * 64)
print("T3 — Comprobación empírica (top-p p=0.9, T=1)")
print("=" * 64)
print(f"Generador: numpy.random.default_rng({SEMILLA}) | Muestras: {N_MUESTRAS}")
print(f"Distribución teórica: {fuente}")
print(f"Tokens del núcleo: {tokens_sobrevivientes}")
print("-" * 64)
print(f"{'Token':<6}{'Teórica':>12}{'Observada':>12}{'Conteo':>10}{'|diff|':>12}")
for t, th, e, c in zip(TOKENS, p_teorica, p_emp, conteos):
    print(f"{t:<6}{th:>12.6f}{e:>12.6f}{int(c):>10d}{abs(e - th):>12.6f}")
print("-" * 64)
print(f"Divergencia KL(empírica || teórica) = {kl_bits:.6f} bits")
print(f"Entropía empírica = {entropia_emp_bits:.6f} bits")
print(f"Figura guardada: {FIGURA}")
print("Resultados escritos en resultados.json")


T3 — Comprobación empírica (top-p p=0.9, T=1)
Generador: numpy.random.default_rng(0) | Muestras: 10000
Distribución teórica: entrada/T2/resultados.json (distribucion_renormalizada de la Parte 2)
Tokens del núcleo: ['t0', 't1', 't2', 't3']
----------------------------------------------------------------
Token      Teórica   Observada    Conteo      |diff|
t0        0.569376    0.568200      5682    0.001176
t1        0.209462    0.209300      2093    0.000162
t2        0.127045    0.129600      1296    0.002555
t3        0.094117    0.092900       929    0.001217
t4        0.000000    0.000000         0    0.000000
t5        0.000000    0.000000         0    0.000000
----------------------------------------------------------------
Divergencia KL(empírica || teórica) = 0.000050 bits
Entropía empírica = 1.636160 bits
Figura guardada: T3_comparacion_frecuencias.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0**, el argumento z_i/T amplifica las diferencias entre logits: el token de mayor logit (t0, con logit 2.0) conserva un exponente finito mientras que para los demás el exponente tiende a −∞ y su probabilidad a 0. La distribución converge así a un one-hot sobre t0, es decir, a una distribución determinista cuya entropía se anula. Las entropías medidas en la Parte 1 muestran esa tendencia: al bajar T de 2 a 1 y de 1 a 0.5, la entropía cae monótonamente (2.2888 → 1.7998 → 0.9147 bits) y la masa de t0 crece (0.3583 → 0.5516 → 0.8231); cuanto más baja la temperatura, más cerca está la distribución del límite sin aleatoriedad.

**Cuando T → ∞** ocurre lo contrario: z_i/T → 0 para todos los tokens, todas las exponenciales valen lo mismo y las probabilidades se igualan: la distribución tiende a la uniforme sobre los seis tokens. La evidencia medida apunta en esa dirección: al subir T la masa se reparte (a T = 2, t0 baja a 0.3583 y t5 alcanza su máximo de la tabla, 0.0294) y la entropía llega a su mayor valor medido, 2.2888 bits, siguiendo la secuencia creciente 0.9147 < 1.7998 < 2.2888.

**Por qué T → 0 equivale a la decodificación voraz:** dividir los logits por T no altera su orden, de modo que el argmax sigue siendo t0 para cualquier temperatura. La decodificación voraz (greedy) elige en cada paso exactamente ese token de mayor logit, es decir, siempre t0; la softmax con T → 0 concentra toda la probabilidad en ese mismo token, por lo que muestrear de ella devuelve siempre t0. Ambas estrategias producen así la misma secuencia determinista. La entropía lo cuantifica: a T = 0.5 ya hay 0.8231 de masa en t0 con solo 0.9147 bits de incertidumbre, y al reducir T la distribución se aproxima al comportamiento completamente determinista de greedy.